In [1]:
import osmnx as ox
import networkx as nx
from shapely.geometry import Point, LineString, MultiLineString
from shapely.ops import substring
import geopandas as gpd
import numpy as np
import pandas as pd
import folium
from folium import plugins

In [2]:
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')
canopy = gpd.read_file('data/canopy.gdb')
property_full = gpd.read_file('output/hedonic_gdf.gpkg')

pd.set_option('display.max_columns', None)

ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)
boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('spreydon')]

property = gpd.clip(property_full, boundary)
TARGET_CRS = 2193

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [3]:
G = ox.graph_from_place(
  'Christchurch, New Zealand',
  network_type='drive',
  simplify=True,
)

G = ox.project_graph(G, to_crs=TARGET_CRS)

edges = ox.graph_to_gdfs(G, nodes=False)
edges = edges.to_crs(property.crs)

def nearest_street_and_point(point, edges_gdf):
    # find index of nearest street segment
    idx = edges_gdf.geometry.distance(point).idxmin()
    street_geom = edges_gdf.loc[idx].geometry

    # project property onto street
    proj_dist = street_geom.project(point)
    access_point = street_geom.interpolate(proj_dist)

    return street_geom, access_point
  
property[["front_street_geom", "access_point"]] = (
    property.geometry
    .apply(lambda p: nearest_street_and_point(p, edges))
    .apply(pd.Series)
)

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


# 1

In [4]:
def get_street_reach(property_row, property_crs, G, max_dist=200, verbose=False):
    """
    Get all streets within max_dist meters from access point.
    Returns MultiLineString of reachable street segments.
    """
    
    access_pt = property_row['access_point']
    
    G_proj = ox.project_graph(G)
    
    access_gdf = gpd.GeoDataFrame({'geometry': [access_pt]}, crs=property_crs)
    access_proj = access_gdf.to_crs(G_proj.graph['crs']).iloc[0].geometry
    
    nearest_node = ox.nearest_nodes(G_proj, access_proj.x, access_proj.y)
    
    dist_to_nearest = Point(G_proj.nodes[nearest_node]['x'], 
                           G_proj.nodes[nearest_node]['y']).distance(access_proj)
    
    if verbose:
        print(f"Nearest node: {nearest_node}, distance: {dist_to_nearest:.2f}m")
    
    adjusted_radius = max_dist + dist_to_nearest + 50
    
    G_undir = G_proj.to_undirected()
    
    lengths = nx.single_source_dijkstra_path_length(
        G_undir, 
        nearest_node, 
        cutoff=adjusted_radius, 
        weight='length'
    )
    
    if verbose:
        print(f"Reachable nodes from nearest: {len(lengths)}")
    
    reachable_nodes = set(lengths.keys())
    
    lines = []
    for u, v, data in G_undir.edges(data=True):
        if u in reachable_nodes or v in reachable_nodes:
            
            if 'geometry' in data:
                geom = data['geometry']
            else:
                u_pt = Point(G_undir.nodes[u]['x'], G_undir.nodes[u]['y'])
                v_pt = Point(G_undir.nodes[v]['x'], G_undir.nodes[v]['y'])
                geom = LineString([u_pt, v_pt])
            
            dist_to_edge = geom.distance(access_proj)
            
            if dist_to_edge <= max_dist:
                lines.append(geom)
    
    if verbose:
        print(f"Collected {len(lines)} street segments")
    
    if not lines:
        return MultiLineString([])
    
    result = MultiLineString(lines)
    result_gdf = gpd.GeoDataFrame({'geometry': [result]}, crs=G_proj.graph['crs'])
    return result_gdf.to_crs(property_crs).iloc[0].geometry


def calculate_canopy_along_streets(property_row, property_crs, G, canopy_gdf, 
                                   street_dist=200, buffer_dist=20, verbose=False):
    """
    Complete workflow: get street reach, buffer it, calculate canopy area.
    
    Parameters
    ----------
    property_row : Series
        Property with 'access_point' 
    property_crs : CRS
        CRS of property data
    G : networkx graph
        Street network
    canopy_gdf : GeoDataFrame
        Tree canopy polygons
    street_dist : float
        Maximum street distance (meters)
    buffer_dist : float
        Buffer around streets (meters)
    
    Returns
    -------
    dict with 'street_reach', 'buffer_geom', 'canopy_area'
    """
    
    street_reach = get_street_reach(property_row, property_crs, G, street_dist, verbose)
    
    if street_reach.is_empty:
        return {
            'street_reach': street_reach,
            'buffer_geom': None,
            'canopy_area': 0.0
        }
    
    buffer_geom = street_reach.buffer(buffer_dist)
    
    canopy_in_buffer = canopy_gdf[canopy_gdf.intersects(buffer_geom)]
    
    if len(canopy_in_buffer) == 0:
        canopy_area = 0.0
    else:
        canopy_clipped = canopy_in_buffer.intersection(buffer_geom)
        canopy_area = canopy_clipped.area.sum()
    
    if verbose:
        print(f"Canopy area: {canopy_area:.2f} sq meters")
    
    return {
        'street_reach': street_reach,
        'buffer_geom': buffer_geom,
        'canopy_area': canopy_area
    }


def process_all_properties(properties_gdf, G, canopy_gdf=None, 
                          street_dist=200, buffer_dist=20):
    """
    Process all properties to calculate street reach and optionally canopy area.
    
    Parameters
    ----------
    properties_gdf : GeoDataFrame
        Properties with 'access_point' column
    G : networkx graph
        Street network
    canopy_gdf : GeoDataFrame, optional
        Tree canopy data (if None, only calculates street reach)
    street_dist : float
        Street distance (meters)
    buffer_dist : float
        Buffer distance (meters)
    
    Returns
    -------
    GeoDataFrame with added columns
    """
    
    properties = properties_gdf.copy()
    
    street_reaches = []
    buffer_geoms = []
    canopy_areas = []
    
    for idx, row in properties.iterrows():
        if idx % 50 == 0:
            print(f"Processing {idx}/{len(properties)}")
        
        if canopy_gdf is not None:
            result = calculate_canopy_along_streets(
                row, properties.crs, G, canopy_gdf, 
                street_dist, buffer_dist, verbose=False
            )
            street_reaches.append(result['street_reach'])
            buffer_geoms.append(result['buffer_geom'])
            canopy_areas.append(result['canopy_area'])
        else:
            reach = get_street_reach(row, properties.crs, G, street_dist, verbose=False)
            street_reaches.append(reach)
    
    properties['street_reach'] = street_reaches
    
    if canopy_gdf is not None:
        properties['street_buffer'] = buffer_geoms
        properties['canopy_area_200m'] = canopy_areas
    
    return properties

In [5]:
def plot_street_reach_interactive(property_row, reach_geom, property_crs, max_dist=200, save_path='street_reach.html'):
    
    prop_gdf = gpd.GeoDataFrame({'geometry': [property_row.geometry]}, crs=property_crs).to_crs('EPSG:4326')
    access_gdf = gpd.GeoDataFrame({'geometry': [property_row['access_point']]}, crs=property_crs).to_crs('EPSG:4326')
    reach_gdf = gpd.GeoDataFrame({'geometry': [reach_geom]}, crs=property_crs).to_crs('EPSG:4326')
    
    prop_pt = prop_gdf.iloc[0].geometry
    access_pt = access_gdf.iloc[0].geometry
    reach_st = reach_gdf.iloc[0].geometry
    
    center_lat = prop_pt.y
    center_lon = prop_pt.x
    
    m = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=16,
        tiles='OpenStreetMap',
        control_scale=True
    )
    
    folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)
    
    folium.CircleMarker(
        location=[prop_pt.y, prop_pt.x],
        radius=10,
        color='darkgreen',
        fill=True,
        fillColor='lightgreen',
        fillOpacity=0.8,
        popup='<b>Property</b>',
        tooltip='Property Location',
        weight=3
    ).add_to(m)
    
    folium.CircleMarker(
        location=[access_pt.y, access_pt.x],
        radius=7,
        color='darkblue',
        fill=True,
        fillColor='blue',
        fillOpacity=0.9,
        popup='<b>Access Point</b>',
        tooltip='Access Point',
        weight=2
    ).add_to(m)
    
    folium.PolyLine(
        locations=[[prop_pt.y, prop_pt.x], [access_pt.y, access_pt.x]],
        color='blue',
        weight=2,
        opacity=0.6,
        dash_array='5, 10',
        tooltip='Property to Access Point'
    ).add_to(m)
    
    if not reach_st.is_empty:
        reach_json = reach_gdf.to_json()
        
        folium.GeoJson(
            reach_json,
            name=f'Street Reach ({max_dist}m)',
            style_function=lambda x: {
                'color': 'red',
                'weight': 4,
                'opacity': 0.7
            },
            highlight_function=lambda x: {
                'color': 'darkred',
                'weight': 6,
                'opacity': 0.9
            },
            tooltip=f'Streets within {max_dist}m'
        ).add_to(m)
        
        num_segments = len(reach_st.geoms) if hasattr(reach_st, 'geoms') else 1
        total_length = sum(g.length for g in reach_st.geoms) if hasattr(reach_st, 'geoms') else reach_st.length
        
        stats_html = f'''
        <div style="position: fixed; 
                    top: 10px; right: 10px; width: 250px; 
                    background-color: white; z-index:9999; 
                    border:2px solid grey; border-radius: 5px; padding: 10px;
                    box-shadow: 2px 2px 6px rgba(0,0,0,0.3); font-size: 13px;">
            <h4 style="margin: 0 0 10px 0;">Street Reach Stats</h4>
            <p style="margin: 5px 0;"><b>Segments:</b> {num_segments}</p>
            <p style="margin: 5px 0;"><b>Total Length:</b> {total_length:.1f}m</p>
            <p style="margin: 5px 0;"><b>Max Distance:</b> {max_dist}m</p>
        </div>
        '''
        m.get_root().html.add_child(folium.Element(stats_html))
    
    legend_html = f'''
    <div style="position: fixed; 
                bottom: 50px; right: 50px; width: 280px; height: auto; 
                background-color: white; z-index:9999; font-size:13px;
                border:2px solid grey; border-radius: 5px; padding: 12px;
                box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
        <h4 style="margin-top:0; margin-bottom:10px; font-size: 15px;">
            Street Reach ({max_dist}m)
        </h4>
        <p style="margin: 5px 0; line-height: 1.6;">
            <span style="color: lightgreen; font-size: 20px;">●</span> 
            <b>Property</b>
        </p>
        <p style="margin: 5px 0; line-height: 1.6;">
            <span style="color: blue; font-size: 18px;">●</span> 
            <b>Access Point</b>
        </p>
        <p style="margin: 5px 0; line-height: 1.6;">
            <span style="color: red; font-weight: bold; font-size: 16px;">━━</span> 
            <b>Reachable Streets</b>
        </p>
        <hr style="margin: 10px 0;">
        <p style="margin: 5px 0; font-size: 11px; font-style: italic; line-height: 1.4;">
            All streets within {max_dist}m of<br>
            the access point. Use for<br>
            tree canopy buffer analysis.
        </p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    folium.LayerControl(position='topright', collapsed=False).add_to(m)
    
    plugins.Fullscreen(
        position='topleft',
        title='Fullscreen',
        title_cancel='Exit Fullscreen',
        force_separate_button=True
    ).add_to(m)
    
    plugins.MeasureControl(
        position='topleft',
        primary_length_unit='meters',
        secondary_length_unit='kilometers'
    ).add_to(m)
    
    return m

In [9]:
prop = property.iloc[30]
reach = get_street_reach(prop, property.crs, G, max_dist=500, verbose=True)
print(f"Segments: {len(reach.geoms)}")
print(f"Total street length: {sum(g.length for g in reach.geoms):.1f}m")

m = plot_street_reach_interactive(prop, reach, property.crs, max_dist=500)
m

Nearest node: 252148340, distance: 293.63m
Reachable nodes from nearest: 42
Collected 3 street segments
Segments: 3
Total street length: 546.1m
